# Interaction-Energy Particle Flows

This notebook generates `fig:gradflow-interaction-particles`.  For an empirical measure, an interaction energy
$$
    f(\alpha)=\iint k(x,y)\,d\alpha(x)d\alpha(y)
$$
produces coupled ODEs.  We compare three kernels: a repulsive Gaussian, an attractive Gaussian, and a long-range attraction with short-range repulsion.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection

NAME = "gradflow-interaction-particles"
out = figure_dir(NAME)
rng = np.random.default_rng(19)

The examples are schematic and deterministic.  They illustrate the sign and range of the kernel, not solver performance.

In [2]:
n = 30
angles = np.linspace(0, 2*np.pi, n, endpoint=False)
r0 = 0.32 + 0.10 * rng.random(n)
x0 = np.column_stack([r0*np.cos(angles), 0.78*r0*np.sin(angles)])
x0 += 0.045 * rng.normal(size=x0.shape)
steps = 120
dt = 0.050
sigma = 0.36

def gaussian_repulsion(X, strength=0.85):
    diff = X[:, None, :] - X[None, :, :]
    r2 = np.sum(diff**2, axis=-1)
    K = np.exp(-r2 / (2 * sigma**2))
    return strength * (K[:, :, None] * diff / sigma**2).sum(axis=1) / len(X)

def gaussian_attraction(X, strength=0.75):
    return -strength * gaussian_repulsion(X, strength=1.0)

def attraction_repulsion(X):
    center = X.mean(axis=0, keepdims=True)
    long_range_attraction = -0.42 * (X - center)
    short_range_repulsion = 0.55 * gaussian_repulsion(X, strength=1.0)
    return long_range_attraction + short_range_repulsion

def simulate(velocity):
    traj = np.zeros((steps + 1, n, 2))
    traj[0] = x0
    X = x0.copy()
    for k in range(steps):
        V = velocity(X)
        speed = np.linalg.norm(V, axis=1).max()
        if speed > 1e-12:
            V = V / max(speed, 1.8)
        X = X + dt * V
        traj[k+1] = X
    return traj

trajectories = {
    "repulsive": simulate(gaussian_repulsion),
    "attractive": simulate(gaussian_attraction),
    "attraction-repulsion": simulate(attraction_repulsion),
}
all_points = np.concatenate([T.reshape(-1, 2) for T in trajectories.values()], axis=0)
xlim, ylim = padded_limits(all_points, pad=0.08)

In [3]:
def draw_traj(traj, filename):
    fig, ax = plt.subplots(figsize=(2.15, 2.00))
    for i in range(n):
        pts = traj[:, i]
        segments = np.stack([pts[:-1], pts[1:]], axis=1)
        cols = [(*interp_color(k/(steps-1)), 0.38) for k in range(steps)]
        ax.add_collection(LineCollection(segments, colors=cols, linewidths=0.55, zorder=1))
    ax.scatter(traj[0,:,0], traj[0,:,1], s=DIRAC_MARKER_SIZE * 0.48, marker="o", color=RED, edgecolor="none", linewidth=0, zorder=3)
    ax.scatter(traj[-1,:,0], traj[-1,:,1], s=DIRAC_MARKER_SIZE * 0.54, marker="o", color=BLUE, edgecolor="none", linewidth=0, zorder=4)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

for key, traj in trajectories.items():
    draw_traj(traj, f"{key}.pdf")
# Backward-compatible gallery aliases.
draw_traj(trajectories["attraction-repulsion"], "trajectories.pdf")
draw_traj(trajectories["attraction-repulsion"][::30], "snapshots.pdf")